In [5]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import TargetEncoder
import warnings

from src.features import engineer_features
warnings.filterwarnings('ignore')

In [6]:
print("Loading data without global leakage...")
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train['geohash'] = train['geohash'].fillna('Missing')
test['geohash'] = test['geohash'].fillna('Missing')

test['demand'] = -1 
df = pd.concat([train, test], ignore_index=True)

Loading data without global leakage...


In [7]:
print("Applying spatial-temporal features...")
df_engineered = engineer_features(df)

train_df = df_engineered[df_engineered['demand'] != -1].copy()
test_df = df_engineered[df_engineered['demand'] == -1].copy()

X = train_df.drop(columns=['Index', 'demand'])
y = train_df['demand']
X_test = test_df.drop(columns=['Index', 'demand'])

cat_features = ['geohash', 'RoadType', 'Weather']

Applying spatial-temporal features...


In [8]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lgb_test_preds = np.zeros(len(X_test))
cat_test_preds = np.zeros(len(X_test))

print("Initiating Leak-Proof Out-of-Fold Ensemble...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx].copy()
    X_val, y_val = X.iloc[val_idx].copy(), y.iloc[val_idx].copy()
    X_test_fold = X_test.copy()
    
    # --- THE UPGRADE: Strict Out-Of-Fold Target Encoding ---
    # Calculates explicit historical traffic baselines without leaking the answers
    encoder = TargetEncoder(target_type='continuous', random_state=42)
    
    X_train['geo_historical_demand'] = encoder.fit_transform(X_train[['geohash']], y_train)
    X_val['geo_historical_demand'] = encoder.transform(X_val[['geohash']])
    X_test_fold['geo_historical_demand'] = encoder.transform(X_test_fold[['geohash']])
    
    # --- Format for LightGBM ---
    X_train_lgb, X_val_lgb, X_test_lgb = X_train.copy(), X_val.copy(), X_test_fold.copy()
    for col in cat_features:
        X_train_lgb[col] = X_train_lgb[col].astype('category')
        X_val_lgb[col] = X_val_lgb[col].astype('category')
        X_test_lgb[col] = X_test_lgb[col].astype('category')

    lgb = LGBMRegressor(n_estimators=2500, learning_rate=0.02, random_state=42, n_jobs=-1)
    lgb.fit(X_train_lgb, y_train, eval_set=[(X_val_lgb, y_val)], 
            callbacks=[early_stopping(stopping_rounds=100), log_evaluation(period=0)])
    lgb_test_preds += lgb.predict(X_test_lgb) / kf.n_splits
    
    # --- Format for CatBoost ---
    X_train_cat, X_val_cat, X_test_cat = X_train.copy(), X_val.copy(), X_test_fold.copy()
    for col in cat_features:
        X_train_cat[col] = X_train_cat[col].astype(str)
        X_val_cat[col] = X_val_cat[col].astype(str)
        X_test_cat[col] = X_test_cat[col].astype(str)
        
    cat = CatBoostRegressor(iterations=2500, learning_rate=0.03, cat_features=cat_features, random_seed=42, verbose=0)
    cat.fit(X_train_cat, y_train, eval_set=(X_val_cat, y_val), early_stopping_rounds=100)
    cat_test_preds += cat.predict(X_test_cat) / kf.n_splits
    
    print(f"Fold {fold+1} Completed.")

# Robust Blended Predictions
final_predictions = (lgb_test_preds * 0.5) + (cat_test_preds * 0.5)

# Safety clip prevents impossible negative demands
final_predictions = np.clip(final_predictions, a_min=0.0, a_max=None)

submission = pd.DataFrame({
    'Index': test_df['Index'].astype(int),
    'demand': final_predictions
})
submission.to_csv('submission.csv', index=False)
print("submission.csv generated successfully! Ready for upload.")

Initiating Leak-Proof Out-of-Fold Ensemble...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003970 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1807
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 14
[LightGBM] [Info] Start training from score 0.093784
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2496]	valid_0's l2: 0.000792089
Fold 1 Completed.
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of c